In [ ]:
from pathlib import Path
import pandas as pd
import pydicom
import os
import numpy as np
import sys
from datetime import datetime
from scipy.ndimage import zoom, sobel
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from xgboost import XGBClassifier

# ==============================================================================
# PIPELINE OVERVIEW & DOCUMENTATION
# ==============================================================================
print("""
================================================================================
RSNA KNEE ABNORMALITY DETECTION: TIMED EXTRACTION & MODEL OPTIMIZATION
Competition Link: https://www.kaggle.com/competitions/rsna-knee-abnormality-detection

PIPELINE WORKFLOW & RELEVANT STEPS:
1. Load Training Labels Data (LLM Report Labels) & Series Metadata
2. Inspect Label Distribution, Binarize Probabilistic Scores (`y > 0.5`), 
   and Report Post-Conversion Patient Statistics (All 0s vs All 1s)
3. Execute & Time High-Dimensional Feature Extraction (~5,120 Features)
4. Initialize 10-Fold Cross-Validation Splitter 
5. Train XGBoost with Tuned Regularization & Subsampling
6. Evaluate via Macro-Averaged ROC AUC on Test Folds 
7. Output Timing Performance & Top 15 Most Important Features
================================================================================
""")

# ==========================================
# Configuration Constants & File Paths
# ==========================================
TRY_TO_LOAD_DATA_FROM_DISK = False
FEATURES_FILE_PATH = Path("extracted_features_5k.csv")

COMP = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
LLM_LABELS_PATH = Path("/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv")
TRAIN_SERIES_PATH = COMP / "train_series.csv"

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

ID_COL = "StudyInstanceUID"


# ==========================================
# 1. Data Loading & Validation Functions
# ==========================================

def load_datasets():
    train_df = pd.read_csv(LLM_LABELS_PATH).drop(
        columns=[c for c in pd.read_csv(LLM_LABELS_PATH, nrows=1).columns if "__conf" in c or "__verdict" in c]
    )
    train_series_df = pd.read_csv(TRAIN_SERIES_PATH)
    return train_df, train_series_df


def validate_datasets(train_df, train_series_df):
    missing_studies = set(train_df[ID_COL]) - set(train_series_df[ID_COL])
    print(f"1. All train_df StudyInstanceUID accounted for: {len(missing_studies) == 0}")
    print(f"2. Train shape: {train_df.shape} | Train Series shape: {train_series_df.shape}\n")


def inspect_and_summarize_labels(train_df):
    print("The raw LLM report labels contain probabilistic/continuous scores ranging from")
    print("0.08 to 0.94 representing model certainty or soft likelihoods. However, ranking")
    print("metrics like ROC AUC and standard classifiers require strict binary indicators")
    print("for training and evaluation. By applying a threshold of > 0.5, we convert these")
    print("soft scores into clear binary targets (0 = normal/unlikely, 1 = abnormal/likely).\n")
    
    print("================ LABELS DISTRIBUTION & NaN INSPECTION ================")
    summary_data = []
    for label in LABELS:
        nan_count = train_df[label].isna().sum()
        value_counts = train_df[label].value_counts().to_dict()
        summary_data.append({
            "Label": label,
            "NaN_Count": nan_count,
            "Value_Counts": value_counts
        })
    
    summary_df = pd.DataFrame(summary_data)
    for _, row in summary_df.iterrows():
        print(f"{row['Label']:>16} {row['NaN_Count']:10} {row['Value_Counts']}")
    print("=====================================================================\n")


def inspect_binary_labels_and_patient_stats(train_df):
    print("================ POST-CONVERSION (0/1) LABEL & PATIENT STATS ================")
    
    # Create a binary DataFrame for the labels
    df_binary = train_df[LABELS].applymap(lambda x: 1 if pd.notnull(x) and x > 0.5 else 0)
    
    summary_binary_data = []
    for label in LABELS:
        val_counts = df_binary[label].value_counts().to_dict()
        summary_binary_data.append({
            "Label": label,
            "Binary_Counts (0 vs 1)": val_counts
        })
        
    for item in summary_binary_data:
        print(f"{item['Label']:>16} -> {item['Binary_Counts (0 vs 1)']}")
        
    # Calculate patients with all 0s vs all 1s across the 12 clinical targets
    row_sums = df_binary.sum(axis=1)
    all_zeros_count = (row_sums == 0).sum()
    all_ones_count = (row_sums == len(LABELS)).sum()
    
    print("---------------------------------------------------------------------")
    print(f"Total Patients (Studies) Evaluated: {len(df_binary)}")
    print(f"Patients with ALL 0s (completely normal across all 12 targets): {all_zeros_count}")
    print(f"Patients with ALL 1s (positive across all 12 targets): {all_ones_count}")
    print("=====================================================================\n")


# ==========================================
# 2. 5K Feature Extraction with Timing
# ==========================================

BASE_COLS = [
    'img_mean', 'img_std', 'img_median', 'img_iqr', 'img_cv', 
    'img_p1', 'img_p5', 'img_p10', 'img_p25', 'img_p75', 'img_p90', 
    'img_p95', 'img_p99', 'img_max', 'img_min', 'edge_mean', 'edge_std', 
    'edge_max', 'sobel_mean', 'sobel_max'
]
for r in range(10):
    for c in range(10):
        BASE_COLS.extend([f'z_{r}_{c}_mean', f'z_{r}_{c}_std', f'z_{r}_{c}_max'])


def extract_series_pixel_features(series_dir):
    if not os.path.exists(series_dir):
        return None
        
    dcm_files = [os.path.join(series_dir, f) for f in os.listdir(series_dir) if f.lower().endswith('.dcm')]
    if not dcm_files:
        return None
    
    dcm_files.sort()
    step = max(1, len(dcm_files) // 16)
    sampled_files = dcm_files[::step][:16]
    
    slice_features = []
    
    for f_path in sampled_files:
        try:
            ds = pydicom.dcmread(f_path)
            if not hasattr(ds, 'pixel_array'):
                continue
            img = ds.pixel_array.astype(np.float32)
            
            if img.max() > img.min():
                img = (img - img.min()) / (img.max() - img.min())
            
            if img.shape[0] != 120 or img.shape[1] != 120:
                zoom_factors = (120 / img.shape[0], 120 / img.shape[1])
                img = zoom(img, zoom_factors, order=1)

            feat_dict = {
                'img_mean': np.mean(img), 'img_std': np.std(img),
                'img_median': np.median(img),
                'img_iqr': np.percentile(img, 75) - np.percentile(img, 25),
                'img_cv': np.std(img) / (np.mean(img) + 1e-5),
                'img_p1': np.percentile(img, 1), 'img_p5': np.percentile(img, 5),
                'img_p10': np.percentile(img, 10), 'img_p25': np.percentile(img, 25),
                'img_p75': np.percentile(img, 75), 'img_p90': np.percentile(img, 90),
                'img_p95': np.percentile(img, 95), 'img_p99': np.percentile(img, 99),
                'img_max': np.max(img), 'img_min': np.min(img)
            }
            
            gy, gx = np.gradient(img)
            grad_mag = np.sqrt(gx**2 + gy**2)
            sobel_mag = np.sqrt(sobel(img, axis=1)**2 + sobel(img, axis=0)**2)
            
            feat_dict.update({
                'edge_mean': np.mean(grad_mag), 'edge_std': np.std(grad_mag), 'edge_max': np.max(grad_mag),
                'sobel_mean': np.mean(sobel_mag), 'sobel_max': np.max(sobel_mag)
            })
            
            blocks = img.reshape(10, 12, 10, 12).transpose(0, 2, 1, 3)
            block_means = blocks.mean(axis=(2, 3)).flatten()
            block_stds = blocks.std(axis=(2, 3)).flatten()
            block_maxs = blocks.max(axis=(2, 3)).flatten()
            
            for i in range(100):
                r, c = divmod(i, 10)
                feat_dict[f'z_{r}_{c}_mean'] = block_means[i]
                feat_dict[f'z_{r}_{c}_std'] = block_stds[i]
                feat_dict[f'z_{r}_{c}_max'] = block_maxs[i]
            
            slice_features.append(feat_dict)
        except Exception:
            continue
            
    if not slice_features:
        return None
        
    df_slice = pd.DataFrame(slice_features)
    series_agg = {}
    for col in BASE_COLS:
        if col in df_slice.columns:
            series_agg[f'{col}_mean'] = df_slice[col].mean()
            series_agg[f'{col}_max'] = df_slice[col].max()
            series_agg[f'{col}_std'] = df_slice[col].std()
            series_agg[f'{col}_median'] = df_slice[col].median()
        
    return series_agg


def get_or_create_features(train_df, train_series_df):
    if TRY_TO_LOAD_DATA_FROM_DISK and FEATURES_FILE_PATH.exists():
        df_train_features = pd.read_csv(FEATURES_FILE_PATH).set_index(ID_COL)
        print(f"Loaded 5K features from disk. Shape: {df_train_features.shape}")
        return df_train_features, 0.0

    print(f"\nExtracting ~5,120 features via Vectorized 10x10 spatial grids...")
    extraction_start = datetime.now()
    
    study_ids = train_df[ID_COL].unique()
    total_cases = len(study_ids)
    feature_records = []
    
    for i, study_id in enumerate(study_ids, 1):
        if i % 50 == 0 or i == total_cases:
            elapsed_so_far = (datetime.now() - extraction_start).total_seconds()
            rate = i / max(1, elapsed_so_far)
            eta = (total_cases - i) / max(0.1, rate)
            sys.stdout.write(f"\rProcessing case {i}/{total_cases} | Elapsed: {elapsed_so_far:.1f}s | ETA: {eta:.1f}s")
            sys.stdout.flush()
        
        case_series = train_series_df[train_series_df[ID_COL] == study_id]
        series_features_list = []
        
        for _, row in case_series.iterrows():
            series_id = row.get("SeriesInstanceUID", None)
            if series_id:
                series_dir = COMP / "train" / str(study_id) / str(series_id)
                feat = extract_series_pixel_features(str(series_dir))
                if feat is not None:
                    series_features_list.append(feat)
                        
        study_feat = {ID_COL: study_id}
        expected_series_cols = [f"{c}_{agg}" for c in BASE_COLS for agg in ['mean', 'max', 'std', 'median']]
        
        if series_features_list:
            df_s = pd.DataFrame(series_features_list)
            for col in expected_series_cols:
                if col in df_s.columns:
                    vals = df_s[col].astype(float)
                    study_feat[f'{col}_study_mean'] = vals.mean()
                    study_feat[f'{col}_study_max'] = vals.max()
                    study_feat[f'{col}_study_std'] = vals.std()
                    study_feat[f'{col}_study_median'] = vals.median()
                else:
                    study_feat[f'{col}_study_mean'] = 0.0
                    study_feat[f'{col}_study_max'] = 0.0
                    study_feat[f'{col}_study_std'] = 0.0
                    study_feat[f'{col}_study_median'] = 0.0
        else:
            for col in expected_series_cols:
                study_feat[f'{col}_study_mean'] = 0.0
                study_feat[f'{col}_study_max'] = 0.0
                study_feat[f'{col}_study_std'] = 0.0
                study_feat[f'{col}_study_median'] = 0.0
                
        feature_records.append(study_feat)
                
    extraction_end = datetime.now()
    extraction_duration = (extraction_end - extraction_start).total_seconds()
    print(f"\nExtraction complete in {extraction_duration:.2f} seconds ({extraction_duration / 60:.2f} minutes).")
    
    df_train_features = pd.DataFrame(feature_records).set_index(ID_COL).fillna(0)
    df_train_features.to_csv(FEATURES_FILE_PATH)
    return df_train_features, extraction_duration


# ==========================================
# 3. XGBoost Full-Feature Evaluation
# ==========================================

def evaluate_xgboost_all_features(df_train_features, df_train_labels):
    print(f"\n================ MODEL TRAINING (ALL {df_train_features.shape[1]} FEATURES) ================")
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    X = df_train_features.values
    feature_names = df_train_features.columns
    global_importances = np.zeros(len(feature_names))
    
    mean_auc_list = []
    
    for label in LABELS:
        valid_idx = df_train_labels[label].dropna().index
        y = (df_train_labels.loc[valid_idx, label].values > 0.5).astype(int)
        
        if len(np.unique(y)) < 2:
            continue
            
        sub_indices = [df_train_features.index.get_loc(idx_val) for idx_val in valid_idx]
        X_label = X[sub_indices]
        
        fold_preds = np.zeros(len(y))
        fold_trues = np.zeros(len(y))
        label_importances = np.zeros(len(feature_names))
        
        for train_idx, test_idx in kf.split(X_label):
            X_train, X_test = X_label[train_idx], X_label[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            clf = XGBClassifier(
                n_estimators=150,
                learning_rate=0.03,
                max_depth=4,
                colsample_bytree=0.15,
                subsample=0.8,
                reg_alpha=1.0,
                reg_lambda=1.0,
                random_state=42,
                eval_metric='logloss',
                n_jobs=-1
            )
            clf.fit(X_train, y_train)
            
            probas = clf.predict_proba(X_test)[:, 1]
            fold_preds[test_idx] = probas
            fold_trues[test_idx] = y_test
            
            label_importances += clf.feature_importances_ / 10.0
            
        global_importances += label_importances / len(LABELS)
        
        auc = roc_auc_score(fold_trues, fold_preds)
        mean_auc_list.append(auc)
        print(f"[{label.rjust(16)}] 10-Fold AUC: {auc:.4f}")
        
    overall_mean_auc = np.mean(mean_auc_list)
    print(f"--------------------------------------------------")
    print(f"Overall Macro-Averaged AUC: {overall_mean_auc:.4f}")
    
    top_idx = np.argsort(global_importances)[::-1][:15]
    print("\nTop 15 Most Predictive Features:")
    print(f"{'Rank':<4} {'Feature Name':<45} {'Importance Score'}")
    print("-" * 65)
    for rank, i in enumerate(top_idx, 1):
        print(f"{rank:<4} {str(feature_names[i]):<45} {global_importances[i]:.5f}")
        
    return overall_mean_auc


# ==========================================
# Main Execution Entry Point
# ==========================================
if __name__ == "__main__":
    script_start_time = datetime.now()
    train_df, train_series_df = load_datasets()
    validate_datasets(train_df, train_series_df)
    
    # 1. Inspect initial label distribution and probabilistic score thresholding context
    inspect_and_summarize_labels(train_df)
    
    # 2. Inspect post-conversion 0/1 label distribution and full-target patient sums (all 0s / all 1s)
    inspect_binary_labels_and_patient_stats(train_df)

    # 3. Feature Extraction with elapsed timing measurement
    df_train_features, extraction_duration = get_or_create_features(train_df, train_series_df)

    df_train_labels = train_df.set_index(ID_COL)[LABELS]
    df_train_labels = df_train_labels.loc[df_train_features.index]

    # 4. XGBoost Evaluation
    overall_mean_auc = evaluate_xgboost_all_features(df_train_features, df_train_labels)

    script_end_time = datetime.now()
    total_elapsed = (script_end_time - script_start_time).total_seconds()

    print("\n================ FINAL SUMMARY ================")
    print(f"1. Feature Extraction Time: {extraction_duration:.2f} seconds ({extraction_duration / 60:.2f} minutes)")
    print(f"2. Overall Mean AUC: {overall_mean_auc:.4f}")
    print(f"3. Total Pipeline Time: {total_elapsed:.2f} seconds ({total_elapsed / 60:.2f} minutes)")
    print("=============================================")